# Parallel LLM API Calls

This is where async becomes real for agents.
Instead of calling Groq 5 times sequentially,
we call it 5 times simultaneously.

# Setup

In [2]:
import asyncio
import os
import httpx
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {GROQ_API_KEY}",
    "Content-Type": "application/json",
}

# Single async LLM call

In [3]:
async def async_llm_call(client: httpx.AsyncClient, prompt: str) -> str:
    payload = {
        "model": "openai/gpt-oss-20b",
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.7,
        "max_tokens": 200,
    }

    response = await client.post(GROQ_URL, json=payload, headers=headers)
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]

# Test Single Call

In [4]:
async with httpx.AsyncClient(timeout=30) as client:
    result = await async_llm_call(client, "What is Python in one sentence?")
    print(result)

Python is a high‑level, interpreted programming language celebrated for its readability, versatility, and extensive ecosystem of libraries.


# Parallel LLM Calls

In [6]:
async def run_parallel_queries(prompts: list[str]) -> list[str]:
    async with httpx.AsyncClient(timeout=30) as client:
        tasks = [async_llm_call(client, prompt) for prompt in prompts]
        results = await asyncio.gather(*tasks, return_exceptions=True)

    return [
            r if not isinstance(r, Exception) else f"Error {r}"
            for r in results
        ]

topics = [
    "What is LangGraph in one sentence?",
    "What is ChromaDB in one sentence?",
    "What is FastAPI in one sentence?",
    "What is CrewAI in one sentence?",
]

In [7]:
import time
start = time.time()
answers = await run_parallel_queries(topics)
elapsed = time.time() - start

for topic, answer in zip(topics, answers):
    print(f"Q: {topic}")
    print(f"Q: {answer[:100]}")
    print()

print(f"All {len(topics)} queries done in {elapsed:.2f}s")

Q: What is LangGraph in one sentence?
Q: LangGraph is a framework that lets you design and orchestrate language‑model applications as directe

Q: What is ChromaDB in one sentence?
Q: ChromaDB is an open‑source vector database built to efficiently store, index, and retrieve high‑dime

Q: What is FastAPI in one sentence?
Q: FastAPI is a modern, high‑performance Python web framework for building APIs that automatically vali

Q: What is CrewAI in one sentence?
Q: CrewAI is an AI‑powered platform that automates team coordination, workflow management, and decision

All 4 queries done in 1.17s


# Async tool runner (reusable pattern for agents

In [15]:
async def run_tools_parallel(tools: list) -> list:
    tasks = [tool_func(*args) for tool_func, args in tools]
    results = await asyncio.gather(*tasks, return_exceptions=True)
    return results

# Example tools

In [16]:
async def search_web(query: str) -> str:
    await asyncio.sleep(1)   # simulates API call
    return f"Search results for: {query}"

async def read_file(filename: str) -> str:
    await asyncio.sleep(0.5)  # simulates file read
    return f"Contents of: {filename}"

async def fetch_weather(city: str) -> str:
    await asyncio.sleep(0.8)  # simulates weather API
    return f"Weather in {city}: sunny"

In [17]:
tool_calls = [
    (search_web, ["agentic AI trends"]),
    (read_file, ["notes.txt"]),
    (fetch_weather, ["Karachi"]),
]

results = await run_tools_parallel(tool_calls)
for r in results:
    print(r)

Search results for: agentic AI trends
Contents of: notes.txt
Weather in Karachi: sunny
